In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

#Data download
!wget -N 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'

df = pd.read_csv('car_fuel_efficiency_2026.csv')
df.head()

--2026-11-02 22:37:30--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635180 (620K) [text/plain]
Saving to: ‘car_fuel_efficiency_2026.csv’

car_fuel_efficiency 100%[===================>] 620.29K  --.-KB/s    in 0.03s   

Last-modified header missing -- time-stamps turned off.
2026-11-02 22:37:30 (18.8 MB/s) - ‘car_fuel_efficiency_2026.csv’ saved [635180/635180]



,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [2]:
#DATA PREPARATION

#Define categorical and numerical columns
categorical = ["origin","fuel_type","drivetrain"]
numerical = ["model_year","num_doors","engine_displacement","num_cylinders","horsepower","vehicle_weight","acceleration","fuel_efficiency_mpg"]

#Check missing vaalues
print(df.isnull().sum())

#Iterate over each column and fill null values with NA (for string) or 0.0 for float.
for col in df.columns:
    typ = df[col].dtype
    print(typ)
    if str(typ) == "str":
        df[col] = df[col].fillna("NA")
    elif typ == np.float64:
        df[col] = pd.to_numeric(df[col], errors = 'coerce')
        df[col] = df[col].fillna(0.0)
    elif typ == "int64":
        df[col] = pd.to_numeric(df[col], errors = 'coerce')
        df[col] = df[col].fillna(0.0)
        
        
    #print(col)
    #print(df[col].unique()[:5]) #returns unique value in column (first five)
    #print(df[col].nunique()) #returns numnber of unique values
    #print(df[col].count()) #returns count of non-null values
    #print()

print(df.isnull().sum())
    

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64
int64
str
str
str
int64
int64
int64
float64
int64
float64
float64
model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64


In [3]:
#Split Data
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [4]:
#Reset index
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

#Set Y
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

#Delete Y from input (X) matrices
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [5]:
#Turn data frames into matrices
from sklearn.feature_extraction import DictVectorizer
train_dicts = df_train.to_dict(orient='records')
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)

In [6]:
#Question 1- Train Decision Tree Regressor
from sklearn.tree import DecisionTreeRegressor
dtR = DecisionTreeRegressor(max_depth=1)
dtR.fit(X_train, y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,1
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [7]:
#Check the featured considered by model
from sklearn.tree import export_text
print(export_text(dtR, feature_names=list(dv.get_feature_names_out())))
#Answer - model_year

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [8]:
#Question 2- Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

#Train model
randForestModel = RandomForestRegressor(n_estimators=10,random_state=1,n_jobs=-1)
randForestModel.fit(X_train, y_train)

#Transform to matrix val data
val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

#Predict Y based on VAl data
y_pred = randForestModel.predict(X_val)

print(np.sqrt(mean_squared_error(y_val, y_pred)))

#Answer - 1.837

1.837054054730018


In [9]:
#Question 3 - Random Forest Regressor with different n_estimators

nEstimatorList = [10, 50, 100, 150]

for n in nEstimatorList:
    randForestModel = RandomForestRegressor(n_estimators=n,random_state=1,n_jobs=-1)
    randForestModel.fit(X_train, y_train)
    y_pred = randForestModel.predict(X_val)
    print(f"n_estimator: {n}")
    print(f"rmse:{np.sqrt(mean_squared_error(y_val, y_pred)):.3f}")
    print()


n_estimator: 10
rmse:1.837

n_estimator: 50
rmse:1.772

n_estimator: 100
rmse:1.768

n_estimator: 150
rmse:1.769



In [10]:
#Question 4 - Random Forest Regressor with different n_estimators

maxDepthList = [10, 15, 20, 25]
nEstimatorList = [10, 50, 100, 150]



for d in maxDepthList:
    rmse = []
    for n in nEstimatorList:
        randForestModel = RandomForestRegressor(n_estimators=n,max_depth=d,random_state=1,n_jobs=-1)
        randForestModel.fit(X_train, y_train)
        y_pred = randForestModel.predict(X_val)
        rmse.append(np.sqrt(mean_squared_error(y_val, y_pred)))
  
    print(f"max_depth: {d}; mean rmse:{np.mean(rmse):.3f}")
    print()
#Answer: 10

max_depth: 10; mean rmse:1.755

max_depth: 15; mean rmse:1.784

max_depth: 20; mean rmse:1.788

max_depth: 25; mean rmse:1.786



In [12]:
#Question 5
randForestModel = RandomForestRegressor(n_estimators=n,max_depth=d,random_state=1,n_jobs=-1)
randForestModel.fit(X_train, y_train)

for i in range(len(randForestModel.feature_importances_)):
    print(f'{dv.get_feature_names_out()[i]}: {randForestModel.feature_importances_[i]:.3f}')
#Answer - vehicle_weight

acceleration: 0.056
drivetrain=All-wheel drive: 0.008
drivetrain=Front-wheel drive: 0.020
drivetrain=Rear-wheel drive: 0.005
engine_displacement: 0.064
fuel_type=Diesel: 0.017
fuel_type=Gasoline: 0.156
fuel_type=Hybrid: 0.020
horsepower: 0.076
model_year: 0.295
num_cylinders: 0.007
num_doors: 0.019
origin=Asia: 0.006
origin=Europe: 0.004
origin=USA: 0.042
vehicle_weight: 0.203


In [13]:
#Question 6
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 32.2 MB/s  0:00:01s eta 0:00:01:0103
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 25.6 MB/s  0:00:10 eta 0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]━━━ 1/2 [xgboost]

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: /usr/local/python/3.14.2/bin/python3 -m pip install --upgrade pip


In [14]:
#Eta 0.3
import xgboost as xgb

features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
watchlist = [(dtrain, 'train'), (dval, 'val')]
model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist)
y_pred = model.predict(dval)
print(np.sqrt(mean_squared_error(y_val, y_pred)))

[0]	train-rmse:2.40653	val-rmse:2.46782
[1]	train-rmse:2.07098	val-rmse:2.15784
[2]	train-rmse:1.86454	val-rmse:1.97398
[3]	train-rmse:1.73360	val-rmse:1.86662
[4]	train-rmse:1.65059	val-rmse:1.80312
[5]	train-rmse:1.59328	val-rmse:1.76216
[6]	train-rmse:1.55435	val-rmse:1.74392
[7]	train-rmse:1.52677	val-rmse:1.73976
[8]	train-rmse:1.50265	val-rmse:1.73478
[9]	train-rmse:1.48761	val-rmse:1.73282
[10]	train-rmse:1.47125	val-rmse:1.73205
[11]	train-rmse:1.45902	val-rmse:1.73227
[12]	train-rmse:1.44829	val-rmse:1.73503
[13]	train-rmse:1.43604	val-rmse:1.73466
[14]	train-rmse:1.42972	val-rmse:1.73448
[15]	train-rmse:1.41908	val-rmse:1.73512
[16]	train-rmse:1.41206	val-rmse:1.73297
[17]	train-rmse:1.40940	val-rmse:1.73517
[18]	train-rmse:1.40278	val-rmse:1.73584
[19]	train-rmse:1.39637	val-rmse:1.73664
[20]	train-rmse:1.39368	val-rmse:1.73766
[21]	train-rmse:1.39254	val-rmse:1.73701
[22]	train-rmse:1.37639	val-rmse:1.74216
[23]	train-rmse:1.36470	val-rmse:1.74363
[24]	train-rmse:1.36071	va

In [15]:
#Eta 0.1
xgb_params = {
    'eta': 0.1, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
watchlist = [(dtrain, 'train'), (dval, 'val')]
model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist)
y_pred = model.predict(dval)
print(np.sqrt(mean_squared_error(y_val, y_pred)))

#Answer - ETA 0.1

[0]	train-rmse:2.74739	val-rmse:2.78635
[1]	train-rmse:2.58527	val-rmse:2.63162
[2]	train-rmse:2.44540	val-rmse:2.50036
[3]	train-rmse:2.32187	val-rmse:2.38568
[4]	train-rmse:2.21445	val-rmse:2.28765
[5]	train-rmse:2.12109	val-rmse:2.20428
[6]	train-rmse:2.03996	val-rmse:2.13241
[7]	train-rmse:1.96795	val-rmse:2.06798
[8]	train-rmse:1.90640	val-rmse:2.01618
[9]	train-rmse:1.85240	val-rmse:1.96998
[10]	train-rmse:1.80592	val-rmse:1.93152
[11]	train-rmse:1.76410	val-rmse:1.89688
[12]	train-rmse:1.72781	val-rmse:1.86726
[13]	train-rmse:1.69564	val-rmse:1.84312
[14]	train-rmse:1.66745	val-rmse:1.82218
[15]	train-rmse:1.64286	val-rmse:1.80346
[16]	train-rmse:1.62187	val-rmse:1.79038
[17]	train-rmse:1.60321	val-rmse:1.77726
[18]	train-rmse:1.58694	val-rmse:1.76827
[19]	train-rmse:1.57240	val-rmse:1.75951
[20]	train-rmse:1.55847	val-rmse:1.75076
[21]	train-rmse:1.54640	val-rmse:1.74347
[22]	train-rmse:1.53643	val-rmse:1.73912
[23]	train-rmse:1.52675	val-rmse:1.73485
[24]	train-rmse:1.51763	va